# Lab 08-2 - Decision Trees

This lab covers decision tree classifiers and regressors in `sklearn`. We will also explore using cross-validation to compare the performance of decision trees with $k$-nearest neighbors.

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# KNN will warn you about future updates, but we'll turn these off
import warnings
warnings.simplefilter(action = 'ignore', category = FutureWarning)  

The examples in this lab will use the Wisconsin breast cancer data introduced in a previous lab. These data were collected in a study that explored using machine learning to diagnose breast cancer tissue samples $y$ as either:

- cancerous (a label of `'M'` for malignant) or
- non-cancerous (a label of `'B'` for benign)

i.e. $y$ is binary categorical. The $x$ features in this data set describe the average characteristics (radius, symmetry, etc.) of the cells collected in each tissue sample.

In [ ]:
## Read the data
wbc = pd.read_csv(filepath_or_buffer = "../data/wisc_bc.csv")

# Notice the names of the 10 predictor variables
wbc.head(n = 5)

To begin, we'll perform an 80-20 training-testing split, separate the outcome variable, and when creating the feature variables drop the `ID` column as well since it is an identification variable not an actual variable of information:

In [ ]:
## Train-test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(wbc, test_size = 0.2, random_state = 7)

## Separate the target/outcome variable from the predictors
## Then for y variables, change the 'M' vs 'B' coding of binary
## to 1 vs 0
train_y = train['Label'].map({'M': 1, 'B': 0})
test_y = test['Label'].map({'M': 1, 'B': 0})

train_X = train.drop(labels = ['ID','Label'], axis = 1)
test_X = test.drop(labels = ['ID','Label'], axis = 1)

# Take a look
test_y.head(n = 5)

Unlike last time, notice that we re-coded the labels using the `map()` method as follows:

* $y=1$: `M` malignant (cancerous) or 
* $y=0$: `B` benign (non-cancerous)

As we'll soon see, this recoding is to help us keep track of the predicted classes in our decision tree, which uses a generic plotting method that labels the predicted classes using integers.


## A) Decision Tree Classifiers

Depending on the outcome variable $y$ type, we use different modeling objects:

- For classification tasks, we use the `DecisionTreeClassifier()` function to fit/train a decision tree
- For regression tasks, we use the `DecisionTreeRegressor()` function.

The example below fits a tree with a maximum depth of 2 to our training data:

In [ ]:
## Train the tree
from sklearn.tree import DecisionTreeClassifier
my_tree = DecisionTreeClassifier(max_depth = 2, random_state = 1).fit(X = train_X, y = train_y)

Compared with other machine learning algorithms, decision trees are highly interpretable. We can see the structure of the tree using the `plot_tree()` function:

In [ ]:
## Display the fitted tree's structure
from sklearn.tree import plot_tree

plot_tree(
    decision_tree = my_tree,
    # We need to convert the predictor variable names to a list:
    feature_names = train_X.columns.tolist(), 
    # Recall 0 maps to B for benign, 1 maps to M for malignant:
    class_names = ['B', 'M']
)
plt.show()

In a tree visualization, the terminal nodes are at the very bottom of the tree; here we have 4 terminal nodes. In this visualization, the top-line of each non-terminal node shows the splitting rule used to define the subsequent child nodes:

The first split uses the feature/predictor variable `Concave` to split the data into two segments, the left segment being characterized by `Concave <= 0.051`
 
* The samples with `Concave <= 0.051` (left branch) are further split using the rule `Radius <= 15.05`
* The samples with `Concave > 0.051` (right branch) are further split using the rule `Texture <= 16.795`
    
The visualization also includes information on the Gini impurity, the total number of samples present, and the class distribution of each node. For example: the bottom left-most terminal node has:

- 260 observations in it, of which
    - 252 are $y=0$ benign and 
    - 8 are $y=1$ malignant
- a Gini index of 0.06 indicating a low level of impurity i.e. high level of purity (homogeneity)
- a predicted class of $y=0$ B (benign), since that is the majority class

## B) Question 1

Use the tree visualization provided above to answer the following questions.

- **Part a)**: Calculate the Gini gain resulting from the tree's first split. Do not use a python function, rather show your work using arithmetic and functions like `pow()`
- **Part b)**: What is the highest predicted probability for the class "malignant" that can be assigned by this tree? 
- **Part c)**: Consider the first observation *in the test set*, what is the predicted probability that this sample is "malignant"? Hint: You don't need any information other than the contents of the first row of `test_X` and the visualization of the tree above.

In [ ]:
# Code for Q1.a) Change all instances of 76 with the appropriate computation

top_node = 76
print("Top node gini impurity: " + str(top_node))

left_node = 76
print("Left child node gini impurity: " + str(left_node))

right_node = 76
print("Right child node gini impurity: " + str(right_node))

weighted_average = 76*left_node + 76*right_node
print("Weighted average of child node gini impurities: " + str(weighted_average))

gini_gain = top_node - weighted_average
print("Gini gain i.e. the drop in gini impurity: " + str(gini_gain))

**Part b)**: 

In [ ]:
# Code for Q1.c)


**Part c)**:



## C) Hyperparameter Tuning

Decision trees are prone to overfitting, making hyperparameter tuning a very important consideration when training a tree that will be used on new data.

The two most important hyperparameters in regard to potential overfitting are:

1. `max_depth`: the maximum depth of the tree i.e. the most consecutive splits allowed in a single branch
1. `min_samples_split`: the minimum number of observations that must be present in a node for it to be eligible for splitting

In `sklearn`, the default values of:

- `max_depth` is `None`, so the algorithm will continue to split until all of its terminal nodes are pure or contain fewer observations than the value of `min_samples_split`
- `min_samples_split` is 2, which leads to the default set of parameters learning the deepest possible tree, which is not ideal for performance on new data

An alternative to controlling the complexity of the tree using `max_depth` and `min_samples_split` is tuning via the `min_impurity_decrease` parameter, which is the minimum *weighted* improvement in gini impurity needed for a split to occur. The weighting is the fraction of all training observations in that node, so splits on small nodes deep in the tree need a much larger raw gini gain to qualify. You should recognize the following before relying upon this parameter:

1. The final tree will be very data dependent in the sense that you cannot reliably anticipate its depth/complexity ahead of time.
2. Tuning this parameter is more "context sensitive" in the sense that a "hard" classification task will require a more relaxed value to produce a tree with any depth to it.

Regardless of which parameters you choose to tune, cross-validation (using either grid or random search) should be used to find suitable values:

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline 

pipe = Pipeline(steps = [
    ('model', DecisionTreeClassifier(random_state = 1))
])

## Parameters to try via grid search. Observe we are searching over
## 3 x 3 = 9 possible combinations of parameters and trying to 
## identify the best.
params = {
    'model__max_depth': [2,3,4],
    'model__min_samples_split': [10,20,40]
}

## Conduct grid search
grid_results = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = 5, 
    # Note the performance metric/score:
    scoring = 'accuracy'
    ).fit(X = train_X, y = train_y)

## Print results for all 9 parameter combinations as a data frame. The
## estimated accuracy is in the `mean_test_score` column
pd.DataFrame(grid_results.cv_results_)

Now how do we access the results in `grid_results`? We can pick out the best parameter combination and best `accuracy` score as follows:

In [ ]:
print(grid_results.best_estimator_)
print(grid_results.best_score_)

Here we see that using a maximum depth of 3 and requiring 10 samples to be present in a node to allow it to be split yields a tree with a cross-validated classification accuracy of 90.33%.

Note that `.best_estimator_` is the best full pipeline, already refit on the full training data. You can call `.predict()` on it directly without redefining your model, or reuse it as the `estimator` in a later `GridSearchCV`:

In [ ]:
# Simple example use: Use this estimator to make predictions on new data.
# Note best_pipe has already been refit on all of train_X by GridSearchCV
best_pipe = grid_results.best_estimator_
print("Prediction for first test set observation: ", best_pipe.predict(X = test_X)[0])

`best_pipe` is a `Pipeline`, not a tree: it's a container holding a list of named steps. To pull out a single step, use square brackets with the name we gave that step when building the pipeline above:

```python
Pipeline(steps = [('model', DecisionTreeClassifier())])
```

This is like pulling a column out of a data frame by its name, e.g. `train['Label']`. We'll need this whenever a function wants the tree itself, like `plot_tree()`.

In [ ]:
best_tree = best_pipe['model']
print(best_tree)

## D) Question 2

Continue using the Wisconsin breast cancer data for the following questions.
 
- **Part a)**: Use a cross-validated grid search to tune the parameter `min_impurity_decrease` and save the results of `GridSearchCV()` in an object called `grid_results_tree`. Report the classification accuracy of the best estimator.
- **Part b)**: Use `plot_tree()` to visually compare the best estimator you found in Part a) to the best estimator found in the example above. How are these trees similar or different?
- **Part c)**: Add standardization to your data preparation pipeline, then use a cross-validated grid search to select an appropriate $k$-nearest neighbors model. Save the results of `GridSearchCV()` in an object called `grid_results_knn`. You should explore a few different choices of hyperparameters including the number of neighbors and weighting scheme.
- **Part d)**: Look at the example code that uses pipelines in a new way. Answer:
    1. What is this pipeline doing?
    1. What is the output telling us?
- **Part e)**: Find the classification accuracy of each model you considered in Part d) on the test set. Are your findings consistent with the results of Part d)?

In [ ]:
# Code for Q2.a)

In [ ]:
# Code for Q2.b)


In [ ]:
# Code for Q2.c)


In [ ]:
# Code for Q2.d): Example code
# Set up pipeline
pipe_q2d = Pipeline(steps = [
    ('model', grid_results_tree.best_estimator_)
])

# Set up search grid
candidate_models = {
    'model': [grid_results_tree.best_estimator_, grid_results_knn.best_estimator_]
}

# Run GridSearchCV
grid_results_q2d = GridSearchCV(
    estimator = pipe_q2d, 
    # Note the search grid:
    param_grid = candidate_models, 
    cv = 5, 
    scoring = 'accuracy'
    ).fit(X = train_X, y = train_y)

# Print output
print(grid_results_q2d.best_estimator_)
print(grid_results_q2d.best_score_)

**Part d)**:

1. 
1. 

In [ ]:
# Code for Q2.e)


## E) Decision Tree Regressors

The regression examples in this lab will use the Iowa City home sales data set that we've also worked with in several of our previous labs.

For illustration purposes, we'll consider only two features, `'area.living'` and `'bedrooms'` and explore how they can be used to predict a home's sale price: 

In [ ]:
## Read IC home sales data and split into training/test sets
ic = pd.read_csv(filepath_or_buffer = "../data/IowaCityHomeSales.csv")
train_ic, test_ic = train_test_split(ic, test_size = 0.2, random_state = 7)

## Create X and y
train_ic_y = train_ic['sale.amount']
train_ic_X = train_ic[['area.living','bedrooms']]

To begin, you should recognize that the `DecisionTreeRegressor()` function behaves almost exactly the same as `DecisionTreeClassifier()`, but there are a few key differences that we can see in the visualization created by `plot_tree()`:

In [ ]:
## Train the tree
from sklearn.tree import DecisionTreeRegressor
my_tree = DecisionTreeRegressor(max_depth = 2, random_state = 1).fit(X = train_ic_X, y = train_ic_y)

## Visualize tree
plt.figure(figsize = (8,4))
plot_tree(decision_tree = my_tree, feature_names = train_ic_X.columns.tolist())
plt.show()

Observe that
1. The predicted value in each terminal node is the mean of the outcome variable $y$ values for the data-points contained in that node.
1. Splitting rules are learned based upon the drop in squared error (not the drop in gini impurity). This can be consequential for hyperparameter tuning because the units of squared error are related to the units of the outcome variable. However, we can avoid needing to worry about this by tuning our trees using the parameters `max_depth` and `min_samples_split` instead of the parameter `min_impurity_decrease`.

## F) Question 3 (Optional bonus)

Be sure that you're using the Iowa City home sales data for the following questions.

- **Part a)**: Using the training data, `train_ic_X`, create a scatterplot displaying a home's living area on the x-axis and its sale amount on the y-axis.
- **Part b)**: Notice that all splitting rules in the example tree given above involve the predictor `area.living`. For this question, use `hlines()` with the argument `color = 'red'` to display this tree's *predicted values* for different values `area.living`. *Hint*: your graph should display several horizontal lines, similar to the "resting metabolism" example in our lecture slides. You should consult the [hlines documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hlines.html) for additional information on how to use this function.
- **Part c)**: Simple linear regression (using only `area.living` as the predictor) would produce predicted values that follow the straight line that best fits these data. Using your plot from Part b), what advantages and disadvantages do you see in the decision tree model when compared with simple linear regression?

In [ ]:
# Code for Q3.a)


In [ ]:
# Code for Q3.b)


**Part c)**: 

* Advantage: 
* Disadvantage: 
